# Context Management: The On-Call Support Agent

This notebook shows how NOOA agents manage what the LLM sees on every turn — via an automatic event history, static and dynamic context blocks pinned into the system prompt, and (optionally) letting the LLM edit its own context from inside CodeAct.

> **One-liner:** context is an API, not a template. The agent can inspect and mutate
> what the LLM sees, and — if you let it — so can the LLM itself.

This notebook proves four things:

1. **Event history is free.** Every call to a generation method appends events to
   `self.events`. Subsequent calls on the same agent see them. No manual
   `messages=[...]` threading.
2. **Static context blocks pin a value once** into the system prompt.
3. **Dynamic context blocks re-evaluate every LLM turn**, so the LLM always
   sees the freshest state without you rewiring anything.
4. **The LLM can own its own context.** By opting the `context` skill in, the
   support agent can write notes to her own system prompt from inside CodeAct.

We'll finish with a pointer at the two escape hatches for when context alone
is not enough: history summarization and long-term memory.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) — those need no API key, just an `api_base`.

In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# NOOA works with any LiteLLM-supported model — hosted or local.
# Pick one below. Replace "your-api-key" with a real key for hosted providers;
# local providers (Ollama, vLLM) don't need a key — just pass api_base.

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/nvidia/openai/gpt-oss-20b", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)

## Meet the support agent

Two generation methods. Nothing else — no memory store, no chat history object,
no message threading. Just two ellipsis bodies and a class docstring that sets
the character.

In [ ]:
from nooa import Agent, print_prompt
from nooa.agentdoc import hidden, spec

class SupportAgent(Agent, llm=model):
    """You are a customer support agent for a SaaS product.
    You are calm, empathetic, and precise. Keep replies brief."""

    async def handle(self, user: str, issue: str) -> str:
        """Handle `user`'s report: `issue`. Reply in one or two sentences."""
        ...

    async def last_issue(self, user: str) -> str:
        """Recall what `user` contacted us about last time. One sentence."""
        ...

### Two customers, then a question

Handle a ticket from Marco. Then one from Ada. Then ask the agent what Marco
contacted us about — without passing anything about Marco to `last_issue`
other than his name. If the framework is doing its job, the agent already
knows.

In [ ]:
agent = SupportAgent()

print("Marco:", await agent.handle("Marco", "my API key stopped working this morning"))
print()
print("Ada:  ", await agent.handle("Ada", "how do I export invoices as CSV?"))
print()
print("Recall Marco:", await agent.last_issue("Marco"))

## The event history is free

Every generation-method call on an agent appends events to `self.events`:
the `Task` (the docstring, resolved), any `LLMOutput` and `PythonOutput` the
LLM produced along the way, and the return value. Subsequent generation
methods on the **same instance** see them by default.

There is no `messages=[...]` argument. There is no "chat history" object you
pass around. The instance IS the conversation state.

### Peek at the tape

Ask `self.events` what happened. Every method call left a trail.

In [ ]:
for e in agent.events.query():
    label = type(e).__name__
    snippet = str(e).replace("\n", " ")[:110]
    print(f"{label:22} {snippet}")

What each type means in one sentence:

- **`Task`** — the user-facing prompt: the method docstring plus the rendered argument values for this call.
- **`LLMOutput`** — what the model wrote back (natural-language reasoning, code, or a final answer).
- **`PythonOutput`** — the result of any code CodeAct executed in its REPL.
- **`AfterAgentCall`** — the framework's bookmark for the end of a generation-method call, carrying its return value.

You can filter these: `agent.events.query(type="Task")` gives just the tasks;
`agent.events.query(query="Marco")` grep the whole tape for a name. See
`doc(agent.events)` for the full query surface.

> **Why this is different:** most agent frameworks make you construct and thread a
> `messages` list into every call — appending user messages, tool results, and
> assistant replies by hand. Here it's a side effect of calling a method on an
> instance. If you want a **fresh context**, make a new instance. If you want to
> **share context** between two tasks, use the same instance. That's the whole API.

## Static blocks: the refund policy

The event history captures what *happened*. What about things that are true
regardless of any specific ticket — the refund policy, the escalation
thresholds, the tone we want reps to strike? Those don't belong in history.
They belong pinned into the **system prompt**, once, so the LLM sees them from
the top on every call.

The framework calls these **static context blocks**. They live on `self.context`
and behave like a dict.

### Pin the refund policy

`set_static` writes a plain value into the cacheable prefix of the system
prompt. It's evaluated once and stays put until you remove it.

Then we use `print_prompt` to see exactly what the LLM will receive — the block
now sits in the system section.

In [ ]:
agent.context.set_static(
    "refund_policy",
    "Refunds within 30 days of purchase, no questions asked. "
    "Between 30 and 90 days: prorated. After 90 days: escalate to a human.",
)

await print_prompt(agent.handle, user="Sam", issue="I want a refund")

Scroll up in the printed prompt and you'll spot a `<refund_policy>` block in
the system section, above the task. That's it — one line of Python, and every
future call to any method on this agent will have the policy in scope.

> **API note.** Plain dict-assignment (`agent.context["k"] = v`) is a shortcut
> for `set_dynamic` — it puts the block in the *volatile* partition. To put
> something in the *static* (cacheable) partition, call `set_static` explicitly.
> The distinction matters when you care about prompt caching and re-evaluation;
> for teaching purposes, think of it as "static = pinned once, dynamic = fresh
> every turn."

## Static vs dynamic

Static blocks are perfect for rules that don't change: policies, personas,
long-lived constraints. But **which services are healthy** changes minute by
minute. If we made service status a static block, we'd have to remember to
overwrite it whenever an incident opens — and the LLM would happily tell a
user with a broken dashboard that everything is fine.

**Dynamic blocks** solve this. Instead of a value, you register an *expression*
(as a string) that the framework re-evaluates on **every LLM turn**. Whatever
that expression returns is what the LLM sees this turn.

### Wire up the status board

We extend `SupportAgent` with a service-status map and a helper that renders
it. `set_dynamic("service_status", "self._render_status()")` tells the
framework: every LLM turn, call `self._render_status()` and inject the result
into the system prompt under the key `service_status`.

In [ ]:
class SupportAgent(Agent, llm=model):
    """You are a customer support agent for a SaaS product.
    You are calm, empathetic, and precise. Keep replies brief."""

    def __init__(self):
        super().__init__()
        self.services = {
            "API": "operational",
            "Dashboard": "operational",
            "Billing": "operational",
            "Webhooks": "operational",
        }

    def _render_status(self) -> str:
        return "\n".join(f"- {name}: {state}" for name, state in self.services.items())

    async def handle(self, user: str, issue: str) -> str:
        """Handle `user`'s report: `issue`. Reply in one or two sentences."""
        ...

    async def last_issue(self, user: str) -> str:
        """Recall what `user` contacted us about last time. One sentence."""
        ...


agent = SupportAgent()
agent.context.set_static(
    "refund_policy",
    "Refunds within 30 days: automatic. 30-90 days: prorated. 90+ days: escalate.",
)
agent.context.set_dynamic("service_status", "self._render_status()")

### Watch the block move

Peek at the outgoing prompt — the `service_status` block should list Billing
as `operational`. Then open a billing incident (`agent.services["Billing"] =
"outage"`) and peek again. The block updated. We didn't call anything on
`self.context` between peeks. The framework re-ran the expression on our
behalf.

In [ ]:
print("=== Before the incident ===")
await print_prompt(agent.handle, user="Sam", issue="my invoice won't load")

In [ ]:
agent.services["Billing"] = "outage"

print("=== After the incident opens ===")
await print_prompt(agent.handle, user="Sam", issue="my invoice won't load")

### Ask about billing anyway

Now actually run `handle`. The agent should notice, from the dynamic block,
that Billing is down — and acknowledge the outage rather than sending Sam on
a debugging goose chase.

In [ ]:
print(await agent.handle("Sam", "my invoice won't load"))

Restore Billing for the rest of the notebook.

In [ ]:
agent.services["Billing"] = "operational" 

> **Why this is different:** in most frameworks, the system prompt is a template
> file plus f-strings, updated with `.format()` before you send a message. Here
> the system prompt is a **live Python object**. You can iterate it, index into
> it, delete keys, replace them with expressions, or hand the whole API over to
> the LLM. There's no template file to keep in sync with the code.

### List, read, remove

`self.context` behaves like a dict. Iterate its keys, read individual blocks,
delete what you no longer need — all normal Python.

In [ ]:
print("Keys currently pinned:", list(agent.context.keys()))
print()
print("refund_policy =", agent.context["refund_policy"])
print()

del agent.context["refund_policy"]
print("After delete:", list(agent.context.keys()))

In [ ]:
# Confirm the block is really gone from the system prompt.
await print_prompt(agent.handle, user="Sam", issue="quick question about pricing")

## Letting the LLM own its own context

So far *we* have been the ones writing to `self.context`. That's fine when the
orchestration is deterministic and the humans in the loop know what to pin.

But a good support agent notices things we can't easily codify — who's about
to churn, who's escalating, who's a VIP asking a small question that could
snowball. It would be nice if she could **jot her own notes into her own
system prompt**, so that next turn she sees them and acts on them.

By default, `self.context` is **hidden from the LLM**. The agent can't see the
API in `doc(self)` and can't touch it from CodeAct. We opt in with one line:
`spec(self, "context", hidden=False)` inside `__init__`. From that moment on,
the LLM sees the ContextApi in its self-doc and can call `self.context[...] =
...` from generated Python.

### Give the agent a notepad

We rebuild `SupportAgent` one more time. Two changes:

1. `spec(self, "context", hidden=False)` in `__init__` — exposes the context API to the LLM.
2. The class docstring tells her what to do with it: if a customer sounds
   frustrated or at risk of churning, save a note under
   `self.context['at_risk']` so the next turn — and every teammate reading
   the transcript — sees it.

In [ ]:
class SupportAgent(Agent, llm=model):
    """You are a customer support agent for a SaaS product.
    You are calm, empathetic, and precise. Keep replies brief.

    If a customer sounds frustrated, threatens to churn, or asks to speak to
    a manager, append their name and a one-line reason to
    self.context['at_risk'] (a list) before replying."""

    def __init__(self):
        super().__init__()
        self.services = {
            "API": "operational",
            "Dashboard": "operational",
            "Billing": "operational",
            "Webhooks": "operational",
        }
        # Expose the context API to the LLM so it can jot notes into its own prompt.
        spec(self, "context", hidden=False)

    def _render_status(self) -> str:
        return "\n".join(f"- {name}: {state}" for name, state in self.services.items())

    async def handle(self, user: str, issue: str) -> str:
        """Handle `user`'s report: `issue`. If they sound frustrated or at
        risk of churning, record it in self.context['at_risk'] first."""
        ...


agent = SupportAgent()
agent.context.set_dynamic("service_status", "self._render_status()")

In the trace viewer, look for the generated Python cell inside `handle`. When the agent decides a customer is at risk, you should see it write `self.context['at_risk'] = ...` before returning. On the next turn, the rendered prompt contains that new `<at_risk>` block.

### A ticket that escalates

Handle four messages from Lou in a row. The tone gets sharper each time.
Somewhere along the way the agent should decide Lou is at risk, note it in
`self.context['at_risk']`, and adjust her replies accordingly. We inspect
the context after the session to see what she wrote.

In [ ]:
tickets = [
    "hey, the webhook retries are firing twice for the same event",
    "still happening — this is breaking our downstream billing",
    "we've lost an hour of revenue to this. is anyone actually looking?",
    "forget it, I'm evaluating other vendors. cancel my plan.",
]

for msg in tickets:
    print(f"Lou: {msg}")
    print("  Agent:", await agent.handle("Lou", msg))
    print()

In [ ]:
print("Context keys the agent has written:", list(agent.context.keys()))
print()

if "at_risk" in agent.context:
    print("at_risk block:", agent.context["at_risk"])
else:
    print("(the agent didn't flag anyone this round)")

> **Why this is different:** in most frameworks the system prompt is fixed at
> agent-construction time. Any "memory" the LLM appears to have across calls
> is an illusion produced by the framework re-serializing chat history behind
> your back. Here the LLM has a **structured, addressable, writable handle** on
> its own system prompt. It's not "hope the model remembers" — it's a real API
> the LLM invokes as normal Python code inside CodeAct.

> **Corollary:** you should be deliberate about *when* you opt in. A misbehaving
> agent that can write to its own system prompt can wedge itself. Start with
> the API hidden (the default) and expose it only when the task genuinely
> benefits — durable per-session notes, incremental plans, at-risk lists.

## When context stops being enough

Context blocks are the day-to-day tool for "what should the LLM see right now."
Two situations push past what they can do alone:

### 1. The event history gets too big

After enough tickets on the same instance, the transcript starts eating tokens.
`TokenBudgetSummarizer` compresses old events into summary events when a token
budget is exceeded — you keep the recent turns verbatim and get a rolling
digest of everything older.

```python
from nooa.agents import TokenBudgetSummarizer
from nooa.config import TokenBudgetConfig

TokenBudgetSummarizer.install(agent, config=TokenBudgetConfig(max_tokens=1000))
```

See `examples/quickstart/09_summarization.py` for a full walk-through.

### 2. You need memory that survives across sessions

Context blocks and events live on the instance. Kill the process, and they're
gone. For durable, cross-session memory — the agent remembering last month's
frequent flyers — the framework ships a memory subsystem with `remember`,
`recall`, and periodic `reflect` consolidation.

```python
from nooa_memory import MemoryManager, MemoryToolsMixin, MemoryConfig

class SupportAgent(MemoryToolsMixin, Agent, llm=llm):
    ...

agent = SupportAgent()
MemoryManager.install(agent, config=MemoryConfig(enabled=True, path="support_memory.db"))
```

See `examples/quickstart/11_memory.py` for the end-to-end tour, including
dedup-on-write, spontaneous association, reflection, and forgetting.

Both are additive — you install them onto an existing agent, no rewrites.

## Recap

Four things the support agent taught us:

- **Event history is automatic.** Every generation-method call appends to `self.events`; the next call on the same instance sees it. No `messages=[...]` threading.
- **Static context blocks pin values once** (`self.context.set_static("k", v)`). Perfect for policies, personas, escalation rules.
- **Dynamic context blocks re-evaluate every turn** (`self.context.set_dynamic("k", "self.render()")`). Perfect for live state — service status, queue depth, current step of a plan.
- **The LLM can own its own context** if you opt in with `spec(self, "context", hidden=False)`. From inside CodeAct, generated code can read, write, and remove context blocks — the LLM literally editing its own system prompt.

And when context stops being enough: `TokenBudgetSummarizer` for compressing history, `MemoryToolsMixin` + `MemoryManager` for durable memory across sessions.

## Exercises

Try these in fresh cells. Each is a small, self-contained edit on the last
`SupportAgent`.

1. **Tone of voice.** Add a static block `tone` — one of `"formal"`,
   `"friendly"`, `"apologetic"` — and confirm the agent's replies change
   accordingly. Change the tone and rerun; the LLM should follow.
2. **Queue depth.** Add a helper method `_queue_level(self) -> str` that
   returns one of `"quiet"`, `"busy"`, `"slammed"` based on a `self.queue`
   int you increment as tickets arrive. Wire it into a dynamic block called
   `queue`. Handle a few tickets and watch the block change between calls.
3. **VIPs.** Give the agent a `vips: dict[str, str]` on `self`, exposed via a
   dynamic block, and adjust the class docstring so that whenever the agent
   spots a plan tier of Enterprise or above she updates `self.vips[user] =
   note` from inside CodeAct. Now `last_issue` should be able to answer from
   `vips` rather than trawling event history.
4. **Compress the tape.** Install `TokenBudgetSummarizer` on a fresh agent
   with a small token budget (say 500), handle ~10 tickets, then inspect
   `agent.events.query()`. What replaced the older `Task`/`LLMOutput` events?